# Columns in and out

Why gender and occupation are not clustering features, why age is, and how much the clusters depend on BMI.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.preprocessing import StandardScaler

from advisor.features import FEATURES

pd.set_option("display.width", 160)

df = pd.read_csv(Path("../../data/processed/sleep_health.csv"))


def fit(columns, k=5):
    X = StandardScaler().fit_transform(df[columns])
    return KMeans(n_clusters=k, random_state=42, n_init=200).fit_predict(X)


base = fit(FEATURES)

## Gender

In [2]:
df.groupby("Gender")[
    [
        "Age",
        "Sleep Duration",
        "Quality of Sleep",
        "Stress Level",
        "Heart Rate",
        "Mean Arterial Pressure",
        "BMI",
    ]
].mean().round(1)

,Age,Sleep Duration,Quality of Sleep,Stress Level,Heart Rate,Mean Arterial Pressure,BMI
Gender,,,,,,,
Female,47.4,7.2,7.7,4.7,69.3,100.9,0.6
Male,37.1,7.0,7.0,6.1,71.1,97.7,0.3


In [3]:
pd.crosstab(df["Gender"], df["Sleep Disorder"])

Sleep Disorder,Insomnia,No disorder,Sleep Apnea
Gender,,,
Female,36,82,67
Male,41,137,11


Women are ten years older on average, and 67 of the 78 apnea cases are women. In this data gender goes with age.

In [4]:
df["Male"] = (df["Gender"] == "Male").astype(int)
with_gender = fit([*FEATURES, "Male"])
pd.concat(
    {
        "without gender": pd.crosstab(base, df["Gender"]),
        "with gender": pd.crosstab(with_gender, df["Gender"]),
    },
    axis=1,
)

without gender      with gender     
Gender         Female Male      Female Male
row_0                                      
0                  20   81          20   82
1                  34    0           0  105
2                  39  106          73    0
3                  60    2          32    0
4                  32    0          60    2

At k = 5 the groups are the same with or without the column, and three of them are almost all women either way, because of the age link. With six groups the column starts to split the groups by gender:


In [5]:
pd.concat(
    {
        "without gender": pd.crosstab(fit(FEATURES, k=6), df["Gender"]),
        "with gender": pd.crosstab(fit([*FEATURES, "Male"], k=6), df["Gender"]),
    },
    axis=1,
)

without gender      with gender     
Gender         Female Male      Female Male
row_0                                      
0                  39  106          34    0
1                  12   41          13   82
2                  32    0          32    0
3                  34    0          33    2
4                  33    2          73    0
5                  35   40           0  105

Without the column, two of the six groups are mixed. With it, every group is one gender and the partition changes (ARI 0.61 with the fit without gender). The column would decide the groups as soon as k moves past 5.

I leave gender out for a product reason: in an app the question is optional and not everyone answers male or female, so the model should not need it. Gender still goes to the language model as context, and I report the gender mix of each group.

With real data, where gender is always known, I would try comparing heart rate and blood pressure with the norm of the same gender before clustering.

## Age

In [6]:
without_age = fit([c for c in FEATURES if c != "Age"])
print("ARI without age:", round(adjusted_rand_score(base, without_age), 2))

ARI without age: 0.99


Dropping age changes almost nothing: the other columns already carry it. Another option is to keep age but remove its effect from heart rate, blood pressure and BMI, so that a 55 year old with the blood pressure of a 55 year old is not treated as high:

In [7]:
from sklearn.linear_model import LinearRegression

adjusted = df[FEATURES].copy()
for column in ["Heart Rate", "Mean Arterial Pressure", "BMI"]:
    trend = LinearRegression().fit(df[["Age"]], df[column]).predict(df[["Age"]])
    adjusted[column] = df[column] - trend
age_adjusted = KMeans(n_clusters=5, random_state=42, n_init=200).fit_predict(
    StandardScaler().fit_transform(adjusted)
)
print("ARI age-adjusted vs base:", round(adjusted_rand_score(base, age_adjusted), 2))
pd.crosstab(base, age_adjusted, rownames=["base"], colnames=["age-adjusted"])

ARI age-adjusted vs base: 0.87


age-adjusted,0,1,2,3,4
base,,,,,
0,0,0,101,0,0
1,0,34,0,0,0
2,145,0,0,0,0
3,0,0,27,35,0
4,0,0,0,0,32


Four groups are unchanged. The group with good habits and a disorder (62 people) splits: 35 stay together and 27 join the stressed group. That is the same cut k = 7 makes (`k.ipynb`). The adjustment does not change the picture and adds a fitted step, so age stays in as a plain column.

## Occupation

In [8]:
pd.crosstab(df["Occupation"], base)

col_0,0,1,2,3,4
Occupation,,,,,
Accountant,6,0,31,0,0
Doctor,35,2,32,2,0
Engineer,2,32,29,0,0
Lawyer,5,0,42,0,0
Manager,1,0,0,0,0
Nurse,5,0,3,33,32
Sales Representative,2,0,0,0,0
Salesperson,32,0,0,0,0
Scientist,4,0,0,0,0


Two groups are nearly one job each, 32 engineers and 32 nurses. Behind them there are 9 and 8 distinct rows.

In [9]:
jobs = pd.get_dummies(df["Occupation"], dtype=int)
X = StandardScaler().fit_transform(pd.concat([df[FEATURES], jobs], axis=1))
with_jobs = KMeans(n_clusters=5, random_state=42, n_init=200).fit_predict(X)
print("ARI with occupation:", round(adjusted_rand_score(base, with_jobs), 2))
pd.crosstab(df["Occupation"], with_jobs)

ARI with occupation: 0.38


col_0,0,1,2,3,4
Occupation,,,,,
Accountant,0,37,0,0,0
Doctor,2,0,2,2,65
Engineer,0,0,2,61,0
Lawyer,0,45,2,0,0
Manager,0,0,1,0,0
Nurse,66,2,1,0,4
Sales Representative,0,0,2,0,0
Salesperson,0,0,32,0,0
Scientist,0,0,4,0,0


With occupation as eleven yes/no columns the groups become jobs. Four occupations have fewer than five people. I leave it out; what it carries about stress and sleep is already in those columns.

## BMI

In [10]:
pd.crosstab(base, df["BMI Category"])

BMI Category,Normal,Obese,Overweight
row_0,,,
0,38,8,55
1,34,0,0
2,144,0,1
3,0,2,60
4,0,0,32


In [11]:
without_bmi = fit([c for c in FEATURES if c != "BMI"])
print("ARI without BMI:", round(adjusted_rand_score(base, without_bmi), 2))
pd.Series(
    {
        "clusters": normalized_mutual_info_score(df["Sleep Disorder"], base),
        "clusters without BMI": normalized_mutual_info_score(df["Sleep Disorder"], without_bmi),
        "BMI category alone": normalized_mutual_info_score(
            df["Sleep Disorder"], df["BMI Category"]
        ),
    },
    name="NMI with Sleep Disorder",
).round(2)

ARI without BMI: 0.89


clusters                0.33
clusters without BMI    0.34
BMI category alone      0.43
Name: NMI with Sleep Disorder, dtype: float64

Four groups out of five are one BMI category. Without BMI the partition keeps an ARI of 0.89, so BMI is not the only thing behind it. On the sleep disorder, BMI alone does better than the clusters (0.43 against 0.33), and the clusters without BMI reach 0.34. I read the disorder check as a modest confirmation, about as much as one column already gives.